# Daily monetization report — notebook-generated

Executed by Celery beat (`run_notebook_report`) via papermill inside the
worker container — `import app.*` and the worker env (DATABASE_URL,
DMR/Cloudflare AI, Slack) are all available.

For each team with connected accounts it:

1. Builds the standard `MonetizationReport` (program gates, live metrics,
   Telegram funnel, insights, LLM strategy sections/actions).
2. **Executes** the actions — real `SCHEDULED` posts through the normal
   content + media + compliance pipeline (`execute` param toggles).
3. Renders the code-path report HTML plus matplotlib charts embedded as
   `cid:` images, writes `monetization.html` / `.txt` / PNGs and a manifest
   the delivery task emails.

Falls back to the code-path monetization report on failure
(`fallback_to_code`). Run by hand: **Run All**.

In [ ]:
# papermill-injected parameters (tagged "parameters")
insight_days = 30
team_id = None            # optional: restrict to one team UUID
execute = True            # schedule report actions as real posts
output_dir = "/notebooks/output"
run_id = "manual"
manifest_path = ""


In [ ]:
import asyncio
import json
import sys
from pathlib import Path

sys.path.insert(0, "/app")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sqlalchemy import func, select

from app.db.session import async_session_maker
from app.models.social_account import SocialAccount
from app.models.user import Team
from app.services.monetization_report import (
    build_monetization_report,
    execute_report_actions,
)

OUT = Path(output_dir)
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Build + execute per team — same data assembly, same executor, same
# learned-hours scheduling as the code path.
async def _build_all():
    reports = []
    async with async_session_maker() as db:
        q = select(Team)
        if team_id:
            q = q.where(Team.id == team_id)
        teams = (await db.execute(q)).scalars().all()
        for t in teams:
            connected = await db.scalar(
                select(func.count(SocialAccount.id)).where(
                    SocialAccount.team_id == t.id,
                    SocialAccount.status == "active",
                )
            )
            if not connected:
                continue
            rep = await build_monetization_report(
                db, team=t, insight_days=insight_days
            )
            if execute:
                await execute_report_actions(db, t, rep)
            reports.append(rep)
    return reports

reports = await _build_all()
for r in reports:
    print(f"{r.team_name}: {len(r.gates)} gates, {len(r.actions)} actions, "
          f"{len(r.scheduled)} scheduled, llm={r.llm_used}")


In [ ]:
def chart_gates(report, path):
    """Gate progress: met/total criteria per account with a program."""
    gated = [g for g in report.gates if g.program]
    if not gated:
        return False
    labels = [f"{g.platform}\n@{g.handle[:18]}" for g in gated]
    met = [g.program.met_count for g in gated]
    total = [len(g.program.criteria) for g in gated]
    fig, ax = plt.subplots(figsize=(8, max(2.5, 0.5 * len(gated) + 1.5)))
    y = range(len(gated))
    ax.barh(y, total, color="#21262d", label="criteria")
    ax.barh(y, met, color="#10b981", label="met")
    ax.set_yticks(list(y), labels=labels, fontsize=8)
    ax.set_xlabel("criteria met")
    ax.legend(fontsize=8)
    ax.set_title("Monetization gate progress")
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)
    return True


def chart_follower_pace(report, path):
    """Theil-Sen follower slope per account — robust to bad readings."""
    pts = [
        (f"{g.platform}\n@{g.handle[:16]}", g.gain_per_day)
        for g in report.gates
        if g.followers is not None
    ]
    if not pts:
        return False
    labels, vals = zip(*pts)
    colors = ["#10b981" if v > 0 else "#ef4444" for v in vals]
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.bar(labels, vals, color=colors)
    ax.axhline(0, color="#30363d", lw=0.8)
    ax.set_ylabel("followers / day (Theil–Sen)")
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    ax.set_title("Follower growth pace — robust slope")
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)
    return True


def chart_telegram(report, path):
    """Telegram funnel: members + DM leads 7d/30d."""
    if report.telegram_members is None and not report.telegram_leads_30d:
        return False
    fig, ax = plt.subplots(figsize=(5, 3))
    labels = ["Members", "DM leads 7d", "DM leads 30d"]
    vals = [
        report.telegram_members or 0,
        report.telegram_leads_7d,
        report.telegram_leads_30d,
    ]
    ax.bar(labels, vals, color=["#229ED9", "#0a7785", "#00d4c8"])
    ax.set_title("Telegram funnel")
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)
    return True


In [ ]:
# Assemble per-team HTML: code-path report HTML + charts as cid: images.
import html as _html

CHART_CAPTIONS = {
    "gates": (
        "Monetization gate progress — criteria met (green) vs total for "
        "each account's creator program. Grey is the remaining distance "
        "to eligibility; programs marked invite-only only fill when the "
        "platform opens access."
    ),
    "pace": (
        "Follower growth pace — Theil–Sen median slope per account "
        "(followers/day), robust to bad scrape readings. Green grows, "
        "red shrinks. Near-zero bars mean the audience is flat, not "
        "necessarily that content is failing."
    ),
    "telegram": (
        "Telegram funnel — channel members vs DM-captured leads over "
        "7 and 30 days. Leads are the monetizable output: the channel "
        "top-fills, the /start DM flow converts."
    ),
}

manifest_reports = []
skipped = []
for idx, rep in enumerate(reports):
    charts = []
    for name, fn in [
        ("gates", chart_gates),
        ("pace", chart_follower_pace),
        ("telegram", chart_telegram),
    ]:
        p = OUT / f"monetization-{run_id}-{idx}-{name}.png"
        if fn(rep, p):
            charts.append((name, p))

    charts_html = ""
    if charts:
        imgs = "".join(
            f'<div style="margin:14px 0">'
            f'<img src="cid:{name}-{idx}" style="max-width:100%">'
            f'<p style="color:#8b949e;font-size:12px;margin:4px 0">'
            f'{_html.escape(CHART_CAPTIONS[name])}</p></div>'
            for name, _ in charts
        )
        charts_html = f'<hr style="border-color:#30363d">{imgs}'

    html_path = OUT / f"monetization-{run_id}-{idx}.html"
    text_path = OUT / f"monetization-{run_id}-{idx}.txt"
    html_path.write_text(rep.to_html() + charts_html)
    text_path.write_text(rep.to_text())

    manifest_reports.append({
        "subject": rep.subject(),
        "html_file": html_path.name,
        "text_file": text_path.name,
        "attachments": [
            {"file": p.name, "cid": f"{name}-{idx}", "mime": "image/png"}
            for name, p in charts
        ],
    })

manifest = {"reports": manifest_reports}
if manifest_path:
    Path(manifest_path).write_text(json.dumps(manifest))
print("reports:", len(manifest_reports), "skipped:", skipped)
